# MLX 最简数据分片

目标：同一份全量数据如何按 worker 切开——**只讲分片**。

```mermaid
flowchart LR
  full["全量 X,y"] --> shard["X[r::N] / y[r::N]"]
  shard --> verify["验证: 互不重叠 + 并集=全集"]
```

- Notebook：单进程模拟 `NUM_SHARDS=3`，验证切片正确，并对每片做一次本地 `mean`。
- 真多进程：同目录 [`dp_train_min.py`](dp_train_min.py)，终端执行 `mlx.launch -n 3 -- python notebooks/runtime/dp_train_min.py`。
- **不在本笔记范围**：梯度、all-reduce、`average_gradients`、参数更新。

依赖：`mlx`、`numpy`、`matplotlib`（仅可视化）。

In [ ]:
import mlx.core as mx

import numpy as np
import matplotlib.pyplot as plt

## 1. 全量数据

`N = 300` 个二维点 `(X, y)`，两类高斯团各约一半；打乱后**尚未分片**。

In [ ]:
N = 300
SEED = 0

mx.random.seed(SEED)
np.random.seed(SEED)

n0, n1 = N // 2, N - N // 2
X0 = mx.random.normal((n0, 2)) + mx.array([-1.2, -0.8])
X1 = mx.random.normal((n1, 2)) + mx.array([1.2, 0.8])
X = mx.concatenate([X0, X1], axis=0)
y = mx.concatenate([mx.zeros((n0,), dtype=mx.uint32), mx.ones((n1,), dtype=mx.uint32)])

perm = mx.array(np.random.permutation(N))
X, y = X[perm], y[perm]
mx.eval(X, y)

print(f"global N={X.shape[0]}, pos={(y == 1).sum().item()}, neg={(y == 0).sum().item()}")

In [ ]:
Xn = np.array(X)
yn = np.array(y)
markers = {0: "o", 1: "^"}
label_colors = {0: "#4c72b0", 1: "#dd8452"}

fig, ax = plt.subplots(figsize=(6, 5))
for lab in (0, 1):
    m = yn == lab
    ax.scatter(
        Xn[m, 0],
        Xn[m, 1],
        c=label_colors[lab],
        marker=markers[lab],
        edgecolors="k",
        linewidths=0.4,
        s=36,
        alpha=0.85,
        label=f"y={lab}",
    )
ax.set_title("全量数据（按 label，尚未分片）")
ax.set_xlabel("x0")
ax.set_ylabel("x1")
ax.legend(loc="best")
ax.set_aspect("equal", adjustable="datalim")
plt.tight_layout()
plt.show()

## 2. 分片

固定 `NUM_SHARDS = 3`，第 `r` 片取 `X[r::3], y[r::3]`（与真分布式里 `X[rank::size]` 同构）。

In [ ]:
NUM_SHARDS = 3
shard_id = np.arange(N) % NUM_SHARDS
shards = [(X[r::NUM_SHARDS], y[r::NUM_SHARDS]) for r in range(NUM_SHARDS)]

for r, (Xr, yr) in enumerate(shards):
    print(
        f"shard {r}: n={Xr.shape[0]}, "
        f"pos={(yr == 1).sum().item()}, neg={(yr == 0).sum().item()}"
    )

## 3. 验证：互不重叠 + 并集 = 全集

用索引集合断言分片正确，再用散点/柱状图肉眼确认。

In [ ]:
index_sets = [set(np.arange(r, N, NUM_SHARDS).tolist()) for r in range(NUM_SHARDS)]
union = set().union(*index_sets)
assert len(union) == N, f"union size {len(union)} != N={N}"
for i in range(NUM_SHARDS):
    for j in range(i + 1, NUM_SHARDS):
        overlap = index_sets[i] & index_sets[j]
        assert not overlap, f"shard {i} ∩ {j} = {overlap}"
print("OK: shards are disjoint and cover [0, N)")

In [ ]:
colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ax = axes[0]
for r in range(NUM_SHARDS):
    for lab in (0, 1):
        m = (shard_id == r) & (yn == lab)
        ax.scatter(
            Xn[m, 0],
            Xn[m, 1],
            c=colors[r],
            marker=markers[lab],
            edgecolors="k",
            linewidths=0.4,
            s=36,
            alpha=0.85,
            label=f"shard{r}/y={lab}",
        )
ax.set_title("颜色=shard，标记=label")
ax.set_xlabel("x0")
ax.set_ylabel("x1")
ax.legend(loc="best", fontsize=8, ncol=2)
ax.set_aspect("equal", adjustable="datalim")

counts = [int(shards[r][0].shape[0]) for r in range(NUM_SHARDS)]
ax = axes[1]
ax.bar([f"shard{r}" for r in range(NUM_SHARDS)], counts, color=colors)
ax.set_ylabel("n")
ax.set_title("各片样本量")
for i, c in enumerate(counts):
    ax.text(i, c + 1, str(c), ha="center")

plt.tight_layout()
plt.show()
print("sum(n) =", sum(counts), "(期望", N, ")")

## 4. 每片本地计算（非训练）

各片只看见自己的子集：对 `Xr` 求均值。这里没有模型、没有梯度。

In [ ]:
for r, (Xr, yr) in enumerate(shards):
    mean_r = mx.mean(Xr, axis=0)
    mx.eval(mean_r)
    print(f"shard {r}: n={Xr.shape[0]}, mean(X)={np.array(mean_r)}")

## 5. 真多进程怎么跑

同目录脚本 [`dp_train_min.py`](dp_train_min.py) 与上面分片同构：

1. `make_data`：构造全量 `(X, y)`
2. `X[rank::size]`：本进程本地分片
3. 打印 `local n` 与 `mean(X)`（无训练）

在仓库根目录：

```bash
mlx.launch -n 3 -- python notebooks/runtime/dp_train_min.py
```

期望：三个进程 `size=3`，各自 `local n ≈ N/3`，三份 `n` 之和为 `N`。